---
title: "Diel Lake Carbon Analysis — v9 All Seasons"
author: "John C. Ayers"
date: "`r Sys.Date()`"
output:
  html_document:
    toc: true
    df_print: paged
  html_notebook:
    toc: true
    toc_float: true
    number_sections: true
    theme: flatly
    highlight: tango
---

<!-- !diagnostics off -->

```{r setup, message=FALSE, warning=FALSE}
# nolint: start
library(tidyverse)
library(lubridate)
library(scales)
library(knitr)
library(kableExtra)

# Suppress R CMD CHECK warnings for tidyverse NSE (non-standard evaluation)
utils::globalVariables(c(
  "Season", "DateTime", "Daylight_flag", "is_night", "block_id", "xmin", "xmax",
  "Parameter", "Value", "Type", "n", "Mean", "SD", "Median", "Min", "Max",
  "Variable", "GPP_observed_mgO2Lh", "GPP_modeled_mgO2Lh",
  "ER_observed_mgO2Lh", "ER_modeled_mgO2Lh", "NEP_mgO2Lh",
  "SI_Calcite", "SIc", "Ca_measured_mgL", "pCO2_uatm", "CO2flux_mmol_m2_d",
  "pH_measured", "pH_modeled", "Temperature_C", "PAR_Wm2",
  "KO2_m_d", "k600_m_d", "CO2aq_mgL", "HCO3_mgL", "CO3_mgL", "DIC_mgL",
  "d13C_DIC_measured", "d13C_DIC_modeled", "d13C_residual",
  "d13C_CO2aq_eq", "d13C_HCO3_eq", "Na_measured_mgL", "Cl_measured_mgL",
  "SI_scaled", "SIc_scaled", "flux_pos", "flux_neg", "y_outgas", "y_ingas"
))
# nolint: end

# Create output directory for plots
dir.create("plots", showWarnings = FALSE)

# Helper: save a ggplot in both PNG and SVG
save_plot <- function(p, name, width = 12, height = 6, dpi = 300) {
  ggsave(file.path("plots", paste0(name, ".png")), p, # nolint: unmatched_fun
         width = width, height = height, dpi = dpi, bg = "white")
  ggsave(file.path("plots", paste0(name, ".svg")), p, # nolint: unmatched_fun
         width = width, height = height, bg = "white")
  invisible(p)
}

# Publication-quality theme (base_size = 14 → ~8 pt at double-column print
# width). Adjust base_size if targeting a different final print size.
theme_pub <- theme_bw(base_size = 14) +
  theme(
    plot.title         = element_text(face = "bold", size = 16),
    axis.title         = element_text(size = 14),
    axis.text          = element_text(size = 12),
    strip.text         = element_text(size = 13, face = "bold"),
    strip.background   = element_rect(fill = "grey92"),
    legend.title       = element_text(size = 13),
    legend.text        = element_text(size = 12),
    legend.key.size    = unit(0.8, "lines")
  )

# Helper: build a data frame of contiguous night-time intervals for
# geom_rect shading. Each row in the source data is assumed to cover one
# hour; rectangles are padded by ±30 min so they span the full hour of
# each timestep.
get_night_rects <- function(data) {
  data |>
    arrange(Season, DateTime) |> # nolint: unmatched_fun
    group_by(Season) |> # nolint: unmatched_fun
    mutate( # nolint: unmatched_fun
      is_night = Daylight_flag == 0, # nolint: object_name
      block_id = cumsum(is_night != lag(is_night, default = FALSE)) # nolint: object_name
    ) |>
    filter(is_night) |>
    group_by(Season, block_id) |> # nolint: object_name
    summarise( # nolint: unmatched_fun
      xmin = min(DateTime) - minutes(30),
      xmax = max(DateTime) + minutes(30),
      .groups = "drop"
    )
}

# Shorthand geom for night shading (drawn first so it sits behind all data)
night_rect <- function(night_df) {
  geom_rect( # nolint: unmatched_fun
    data        = night_df,
    aes(xmin = xmin, xmax = xmax, ymin = -Inf, ymax = Inf), # nolint: unmatched_fun, object_name
    fill        = "grey80",
    alpha       = 0.45,
    inherit.aes = FALSE
  )
}

season_levels <- c("Winter", "Spring", "Summer", "Fall")
season_colors <- c(Winter = "#4575b4", Spring = "#4dac26",
                   Summer = "#d01c8b", Fall   = "#f1a340")
```

---

# Data Import & Preparation

```{r load-data, message=FALSE}
df <- read_csv("Diel_v9_results_allseasons.csv") |>
  mutate(
    DateTime = mdy_hm(DateTime),
    Season   = factor(Season, levels = season_levels)
  )

cat("Rows:", nrow(df), "\n")
cat("Seasons present:", paste(levels(droplevels(df$Season)), collapse = ", "), "\n")
cat("Date range:", format(min(df$DateTime, na.rm = TRUE)),
    "to", format(max(df$DateTime, na.rm = TRUE)), "\n")
glimpse(df)
```

```{r night-rects}
# Pre-compute night intervals once; reused in every time-series plot
night_df <- get_night_rects(df)

# Subsample observed data to one point every four hours (first point of each season
# is always included; row_number() %% 4 == 1 picks rows 1, 5, 9, … per season)
obs_subsample <- df |>
  group_by(Season) |>
  arrange(DateTime) |>
  filter(row_number() %% 4 == 1) |>
  ungroup()
```

```{r param-definitions}
# Numeric parameters to analyse — excludes binary Daylight_flag
num_params <- c(
  "Temperature_C", "pH_measured", "pH_modeled", "pCO2_uatm",
  "Ca_measured_mgL", "Na_measured_mgL", "Cl_measured_mgL", "SI_Calcite",
  "PAR_Wm2", "GPP_observed_mgO2Lh", "GPP_modeled_mgO2Lh",
  "ER_observed_mgO2Lh", "ER_modeled_mgO2Lh", "NEP_mgO2Lh",
  "KO2_m_d", "k600_m_d", "CO2flux_mmol_m2_d",
  "CO2aq_mgL", "HCO3_mgL", "CO3_mgL", "DIC_mgL",
  "d13C_DIC_measured", "d13C_DIC_modeled", "d13C_residual",
  "d13C_CO2aq_eq", "d13C_HCO3_eq",
  "SIc"
)

# Human-readable axis labels — uses plotmath expression() for sub/superscripts,
# which renders correctly on all R graphics devices including HTML notebooks.
param_labels <- list(
  Temperature_C          = "Temperature (°C)",
  pH_measured            = "pH (measured)",
  pH_modeled             = "pH (modeled)",
  pCO2_uatm              = expression(pCO[2]~"(μatm)"),
  Ca_measured_mgL        = "Ca (mg/L)",
  Na_measured_mgL        = "Na (mg/L)",
  Cl_measured_mgL        = "Cl (mg/L)",
  SI_Calcite             = "SI Calcite (modeled)",
  SIc                    = "SI Calcite (measured)",
  PAR_Wm2                = expression("PAR (W/m"^2*")"),
  GPP_observed_mgO2Lh    = expression("GPP observed (mg"*O[2]*"/L/h)"),
  GPP_modeled_mgO2Lh     = expression("GPP modeled (mg"*O[2]*"/L/h)"),
  ER_observed_mgO2Lh     = expression("ER observed (mg"*O[2]*"/L/h)"),
  ER_modeled_mgO2Lh      = expression("ER modeled (mg"*O[2]*"/L/h)"),
  NEP_mgO2Lh             = expression("NEP (mg"*O[2]*"/L/h)"),
  KO2_m_d                = expression(KO[2]~"(m/d)"),
  k600_m_d               = expression(k[600]~"(m/d)"),
  CO2flux_mmol_m2_d      = expression(CO[2]~"flux (mmol/m"^2*"/d)"),
  CO2aq_mgL              = expression(CO[2]~"aq (mg/L)"),
  HCO3_mgL               = expression(HCO[3]^"-"~"(mg/L)"),
  CO3_mgL                = expression(CO[3]^"2-"~"(mg/L)"),
  DIC_mgL                = "DIC (mg/L)",
  d13C_DIC_measured      = expression(delta^13*"C DIC measured (‰)"),
  d13C_DIC_modeled       = expression(delta^13*"C DIC modeled (‰)"),
  d13C_residual          = expression(delta^13*"C residual (‰)"),
  d13C_CO2aq_eq          = expression(delta^13*"C"~CO[2]~"aq eq (‰)"),
  d13C_HCO3_eq           = expression(delta^13*"C"~HCO[3]^"-"~"eq (‰)")
)

plabel <- function(p) {
  lbl <- param_labels[[p]]
  if (is.null(lbl)) p else lbl
}

# Observed / modeled column pairs (used for combined time-series & stat tests)
obs_mod_pairs <- list(
  pH = list(
    obs   = "pH_measured",
    mod   = "pH_modeled",
    label = "pH"
  ),
  GPP = list(
    obs   = "GPP_observed_mgO2Lh",
    mod   = "GPP_modeled_mgO2Lh",
    label = expression("GPP (mg"*O[2]*"/L/h)")
  ),
  ER = list(
    obs   = "ER_observed_mgO2Lh",
    mod   = "ER_modeled_mgO2Lh",
    label = expression("ER (mg"*O[2]*"/L/h)")
  ),
  d13C_DIC = list(
    obs   = "d13C_DIC_measured",
    mod   = "d13C_DIC_modeled",
    label = expression(delta^13*"C DIC (‰)")
  )
)

# Columns that will appear in a combined obs+mod time-series plot
combined_cols <- unlist(lapply(obs_mod_pairs, function(x) c(x$obs, x$mod)))

# Special-case parameters with bespoke plots
special_ts <- c("pCO2_uatm", "Ca_measured_mgL", "SI_Calcite", "SIc", "CO2flux_mmol_m2_d")

# Remaining parameters get a standard standalone time-series
standalone_ts <- setdiff(num_params, c(combined_cols, special_ts))
```

---

# Histograms

One histogram panel per season for every numeric parameter.

```{r histograms, fig.width=10, fig.height=5, results='hide', message=FALSE, warning=FALSE}
for (p in num_params) {
  pl <- ggplot(df, aes(x = .data[[p]], fill = Season)) +
    geom_histogram(bins = 30, colour = "white", alpha = 0.85) +
    scale_fill_manual(values = season_colors, drop = FALSE) +
    facet_wrap(~Season, scales = "free_y", drop = FALSE) +
    labs(
      x = plabel(p),
      y = "Count"
    ) +
    theme_pub +
    theme(legend.position = "none")
  print(pl)
  save_plot(pl, paste0("hist_", p), width = 10, height = 5)
}
```

---

# Boxplots by Season

```{r boxplots, fig.width=7, fig.height=5, results='hide'}
for (p in num_params) {
  pl <- ggplot(df, aes(x = Season, y = .data[[p]], fill = Season)) +
    geom_boxplot(outlier.size = 0.9, outlier.alpha = 0.4, linewidth = 0.5) +
    scale_fill_manual(values = season_colors, drop = FALSE) +
    labs(
      x = "Season",
      y = plabel(p)
    ) +
    theme_pub +
    theme(legend.position = "none")
  print(pl)
  save_plot(pl, paste0("box_", p), width = 7, height = 5)
}
```

## Observed vs. Modeled — grouped by Season

Grouped boxplots with Season on the x-axis and fill indicating Observed vs. Modeled.

```{r box-obsmod, fig.width=9, fig.height=5, results='hide'}
obs_mod_fill <- c(Observed = "#e31a1c", Modeled = "#1f78b4")

for (nm in names(obs_mod_pairs)) {
  pair <- obs_mod_pairs[[nm]]

  df_long <- df |>
    select(Season, all_of(c(pair$obs, pair$mod))) |>
    pivot_longer(
      cols      = c(all_of(pair$obs), all_of(pair$mod)),
      names_to  = "Type",
      values_to = "Value"
    ) |>
    mutate(Type = factor(
      case_when(Type == pair$obs ~ "Observed", TRUE ~ "Modeled"),
      levels = c("Observed", "Modeled")
    ))

  pl <- ggplot(df_long, aes(x = Season, y = Value, fill = Type)) +
    geom_boxplot(
      position    = position_dodge(0.8),
      outlier.size  = 0.7,
      outlier.alpha = 0.4,
      linewidth     = 0.5
    ) +
    scale_fill_manual(values = obs_mod_fill, name = NULL) +
    labs(
      x = "Season",
      y = pair$label
    ) +
    theme_pub +
    theme(legend.position = "bottom")

  print(pl)
  save_plot(pl, paste0("box_obsmod_", nm), width = 9, height = 5)
}
```

---

# Summary Statistics

Descriptive statistics for each parameter by season and combined across all seasons.

```{r summary-stats}
# Per-season stats
by_season <- df |>
  select(Season, all_of(num_params)) |>
  pivot_longer(-Season, names_to = "Parameter", values_to = "Value") |>
  group_by(Parameter, Season) |>
  summarise(
    n      = sum(!is.na(Value)),
    Mean   = mean(Value,   na.rm = TRUE),
    SD     = sd(Value,     na.rm = TRUE),
    Median = median(Value, na.rm = TRUE),
    Min    = min(Value,    na.rm = TRUE),
    Max    = max(Value,    na.rm = TRUE),
    .groups = "drop"
  )

# All-seasons combined
all_comb <- df |>
  select(all_of(num_params)) |>
  pivot_longer(everything(), names_to = "Parameter", values_to = "Value") |>
  group_by(Parameter) |>
  summarise(
    n      = sum(!is.na(Value)),
    Mean   = mean(Value,   na.rm = TRUE),
    SD     = sd(Value,     na.rm = TRUE),
    Median = median(Value, na.rm = TRUE),
    Min    = min(Value,    na.rm = TRUE),
    Max    = max(Value,    na.rm = TRUE),
    .groups = "drop"
  ) |>
  mutate(Season = "All")

summary_tbl <- bind_rows(by_season, all_comb) |>
  mutate(Season = factor(Season, levels = c(season_levels, "All"))) |>
  arrange(Parameter, Season) |>
  mutate(across(where(is.numeric) & !n, ~round(., 4)))

summary_tbl |>
  kable(
    caption   = "Descriptive statistics by season and combined (All)",
    col.names = c("Parameter", "Season", "n", "Mean", "SD",
                  "Median", "Min", "Max")
  ) |>
  kable_styling(
    bootstrap_options = c("striped", "hover", "condensed"),
    full_width = FALSE,
    font_size  = 12
  ) |>
  scroll_box(height = "500px")
```

---

# Observed vs. Modeled: Statistical Tests

Paired tests (each row contributes one observed and one modeled value at the same time point).
Normality of paired differences assessed with Shapiro–Wilk; where normal, a paired *t*-test is preferred,
otherwise a Wilcoxon signed-rank test. Significance level α = 0.05.

```{r stat-tests, warning=FALSE}
test_results <- imap_dfr(obs_mod_pairs, function(pair, nm) {
  obs_v <- df[[pair$obs]]
  mod_v <- df[[pair$mod]]
  keep  <- !is.na(obs_v) & !is.na(mod_v)
  obs_v <- obs_v[keep]
  mod_v <- mod_v[keep]
  diffs <- obs_v - mod_v

  # When all paired differences are identical (e.g. all zero), neither
  # shapiro.test() nor wilcox.test() can run: both require variance > 0.
  # Check the full diffs vector (not a sample) so the branch is reliable.
  if (diff(range(diffs)) == 0) {
    tibble(
      Parameter               = nm,
      Obs_column              = pair$obs,
      Mod_column              = pair$mod,
      n_pairs                 = length(diffs),
      SW_W                    = NA_real_,
      SW_p                    = NA_real_,
      Normality               = "Constant diffs (tests N/A)",
      Wilcoxon_V              = NA_real_,
      Wilcoxon_p              = NA_real_,
      t_stat                  = NA_real_,
      t_p                     = NA_real_,
      Preferred_test          = "None",
      Significant             = "No (identical distributions)",
      Mean_diff_obs_minus_mod = 0
    )
  } else {
    # Shapiro-Wilk on differences (max 5 000 obs)
    sw_n      <- min(length(diffs), 5000)
    set.seed(42)
    sw        <- shapiro.test(sample(diffs, sw_n))
    normal    <- sw$p.value > 0.05
    norm_note <- if (normal) "Normal" else "Non-normal"

    wt <- wilcox.test(obs_v, mod_v, paired = TRUE, exact = FALSE)
    tt <- t.test(obs_v,     mod_v, paired = TRUE)

    chosen_p <- if (normal) tt$p.value else wt$p.value

    tibble(
      Parameter               = nm,
      Obs_column              = pair$obs,
      Mod_column              = pair$mod,
      n_pairs                 = length(diffs),
      SW_W                    = round(sw$statistic, 4),
      SW_p                    = round(sw$p.value,   4),
      Normality               = norm_note,
      Wilcoxon_V              = round(wt$statistic, 1),
      Wilcoxon_p              = signif(wt$p.value, 4),
      t_stat                  = round(tt$statistic, 3),
      t_p                     = signif(tt$p.value, 4),
      Preferred_test          = if (normal) "t-test" else "Wilcoxon",
      Significant             = if (chosen_p < 0.05) "Yes ***" else "No",
      Mean_diff_obs_minus_mod = round(mean(diffs), 5)
    )
  }
})

test_results |>
  kable(
    caption = "Paired statistical tests — Observed vs. Modeled (α = 0.05)",
    digits  = 4
  ) |>
  kable_styling(
    bootstrap_options = c("striped", "hover", "condensed"),
    full_width = FALSE
  )
```

---

# Time Series Plots

Grey shading indicates night-time periods (Daylight_flag = 0).

## Observed + Modeled (combined panels)

Observed values plotted as points; modeled values as lines.

```{r ts-obs-mod, fig.width=14, fig.height=6, results='hide'}
for (nm in names(obs_mod_pairs)) {
  pair <- obs_mod_pairs[[nm]]

  pl <- ggplot(df, aes(x = DateTime)) +
    night_rect(night_df) +
    # Observed: subsampled to one point every 4 h; modeled: full-resolution line
    geom_point(
      data = obs_subsample,
      aes(y = .data[[pair$obs]], colour = "Observed"),
      size = 2.5, alpha = 0.8
    ) +
    geom_line(
      aes(y = .data[[pair$mod]], colour = "Modeled"),
      linewidth = 0.8
    ) +
    scale_colour_manual(
      values = c(Observed = "#e31a1c", Modeled = "#1f78b4"),
      name   = NULL
    ) +
    scale_x_datetime(
      date_labels = "%b %d\n%H:%M",
      date_breaks = "4 hours",
      guide       = guide_axis(check.overlap = TRUE)
    ) +
    facet_wrap(~Season, scales = "free_x") +
    labs(
      x = NULL,
      y = pair$label
    ) +
    theme_pub +
    theme(
      legend.position = "bottom",
      axis.text.x     = element_text(angle = 45, hjust = 1)
    )

  print(pl)
  save_plot(pl, paste0("ts_", nm), width = 14, height = 6)
}
```

## GPP, ER, and NEP — combined seasonal panels

GPP (green) and ER (red) show modeled lines with subsampled observed open circles.
NEP (blue) has a single measured series shown as a line.

```{r ts-metabolism, fig.width=14, fig.height=6}
metab_colors <- c(GPP = "#33a02c", ER = "#e31a1c", NEP = "#1f78b4")

# Long format for modeled lines (GPP, ER) and the single NEP series
df_metab_lines <- df |>
  select(Season, DateTime,
         GPP = GPP_modeled_mgO2Lh,
         ER  = ER_modeled_mgO2Lh,
         NEP = NEP_mgO2Lh) |>
  pivot_longer(c(GPP, ER, NEP),
               names_to  = "Variable",
               values_to = "Value") |>
  mutate(Variable = factor(Variable, levels = c("GPP", "ER", "NEP")))

# Long format for subsampled observed open symbols (GPP and ER only)
df_metab_pts <- obs_subsample |>
  select(Season, DateTime,
         GPP = GPP_observed_mgO2Lh,
         ER  = ER_observed_mgO2Lh) |>
  pivot_longer(c(GPP, ER),
               names_to  = "Variable",
               values_to = "Value") |>
  mutate(Variable = factor(Variable, levels = c("GPP", "ER", "NEP")))

p_metab <- ggplot() +
  night_rect(night_df) +
  geom_line(
    data      = df_metab_lines,
    aes(x = DateTime, y = Value, colour = Variable),
    linewidth = 0.8
  ) +
  geom_point(
    data   = df_metab_pts,
    aes(x = DateTime, y = Value, colour = Variable),
    shape  = 1,      # open circle
    size   = 2.5,
    stroke = 0.8
  ) +
  scale_colour_manual(values = metab_colors, name = NULL) +
  scale_x_datetime(
    date_labels = "%b %d\n%H:%M",
    date_breaks = "4 hours",
    guide       = guide_axis(check.overlap = TRUE)
  ) +
  facet_wrap(~Season, scales = "free_x") +
  labs(
    x = NULL,
    y = expression("mg"*O[2]*"/L/h")
  ) +
  theme_pub +
  theme(
    legend.position = "bottom",
    axis.text.x     = element_text(angle = 45, hjust = 1)
  )

print(p_metab)
save_plot(p_metab, "ts_GPP_ER_NEP", width = 14, height = 6)
```

## Standalone parameters

```{r ts-standalone, fig.width=14, fig.height=6, results='hide'}
for (p in standalone_ts) {
  pl <- ggplot(df, aes(x = DateTime, y = .data[[p]], colour = Season)) +
    night_rect(night_df) +
    geom_line(linewidth = 0.7) +
    scale_colour_manual(values = season_colors, drop = FALSE) +
    scale_x_datetime(
      date_labels = "%b %d\n%H:%M",
      date_breaks = "4 hours",
      guide       = guide_axis(check.overlap = TRUE)
    ) +
    facet_wrap(~Season, scales = "free_x") +
    labs(
      x = NULL,
      y = plabel(p)
    ) +
    theme_pub +
    theme(
      legend.position = "none",
      axis.text.x     = element_text(angle = 45, hjust = 1)
    )

  print(pl)
  save_plot(pl, paste0("ts_", p), width = 14, height = 6)
}
```

## pCO₂ with atmospheric reference line

```{r ts-pco2, fig.width=12, fig.height=6}
# Per-season annotation data frame (label appears at start of each season's data)
ann_pco2 <- df |>
  group_by(Season) |>
  summarise(DateTime = min(DateTime, na.rm = TRUE), .groups = "drop") |>
  mutate(y = 425, label = "atmospheric = 425 ppmv")

p_pco2 <- ggplot(df, aes(x = DateTime, y = pCO2_uatm, colour = Season)) +
  night_rect(night_df) +
  geom_line(linewidth = 0.7) +
  geom_hline(
    yintercept = 425, linetype = "dashed",
    colour = "black", linewidth = 0.8
  ) +
  geom_text(
    data        = ann_pco2,
    aes(x = DateTime, y = y, label = label),
    colour      = "black", vjust = -0.5, hjust = 0,
    size        = 4.5,        # ~12 pt at figure size
    inherit.aes = FALSE
  ) +
  scale_colour_manual(values = season_colors, drop = FALSE) +
  scale_x_datetime(
    date_labels = "%b %d\n%H:%M",
    date_breaks = "4 hours",
    guide       = guide_axis(check.overlap = TRUE)
  ) +
  facet_wrap(~Season, scales = "free_x") +
  labs(
    x = NULL,
    y = expression(pCO[2]~"(μatm)")
  ) +
  theme_pub +
  theme(
    legend.position = "none",
    axis.text.x     = element_text(angle = 30, hjust = 1)
  )

print(p_pco2)
save_plot(p_pco2, "ts_pCO2_uatm", width = 12, height = 6)
```

## Ca (mg/L) and SI Calcite — dual y-axis

```{r ts-ca-si, fig.width=12, fig.height=6}
ca_rng <- range(df$Ca_measured_mgL, na.rm = TRUE)
# Expand SI range to cover both modeled (SI_Calcite) and measured (SIc) values
si_rng <- range(c(df$SI_Calcite, df$SIc), na.rm = TRUE)

# Linear transforms: SI <-> Ca primary-axis scale
si_to_ca <- function(x) (x - si_rng[1]) / diff(si_rng) * diff(ca_rng) + ca_rng[1]
ca_to_si <- function(x) (x - ca_rng[1]) / diff(ca_rng) * diff(si_rng) + si_rng[1]

df_ca_si <- df |>
  mutate(
    SI_scaled  = si_to_ca(SI_Calcite),
    SIc_scaled = si_to_ca(SIc)
  )

# Subsampled observed SIc points (every 4 h, matching obs_subsample cadence)
df_ca_si_pts <- obs_subsample |>
  mutate(SIc_scaled = si_to_ca(SIc))

p_ca_si <- ggplot(df_ca_si, aes(x = DateTime)) +
  night_rect(night_df) +
  geom_line(aes(y = Ca_measured_mgL,   colour = "Ca (mg/L)"),
            linewidth = 0.7) +
  geom_line(aes(y = SI_scaled,         colour = "SI Calcite (modeled)"),
            linewidth = 0.7, linetype = "dashed") +
  geom_point(
    data   = df_ca_si_pts,
    aes(y = SIc_scaled, colour = "SI Calcite (measured)"),
    shape  = 1, size = 2.5, stroke = 0.8
  ) +
  scale_y_continuous(
    name     = "Ca (mg/L)",
    sec.axis = sec_axis(~ca_to_si(.), name = "SI Calcite")
  ) +
  scale_colour_manual(
    values = c(
      "Ca (mg/L)"             = "#1b7837",
      "SI Calcite (modeled)"  = "#762a83",
      "SI Calcite (measured)" = "#9970ab"
    ),
    name = NULL
  ) +
  scale_x_datetime(
    date_labels = "%b %d\n%H:%M",
    date_breaks = "4 hours",
    guide       = guide_axis(check.overlap = TRUE)
  ) +
  facet_wrap(~Season, scales = "free_x") +
  labs(x = NULL) +
  theme_pub +
  theme(
    legend.position    = "bottom",
    axis.text.x        = element_text(angle = 30, hjust = 1),
    axis.title.y.right = element_text(colour = "#762a83"),
    axis.text.y.right  = element_text(colour = "#762a83"),
    axis.title.y.left  = element_text(colour = "#1b7837"),
    axis.text.y.left   = element_text(colour = "#1b7837")
  )

print(p_ca_si)
save_plot(p_ca_si, "ts_Ca_SI_Calcite", width = 12, height = 6)
```

## CO₂ flux — outgassing and ingassing areas

```{r ts-co2flux, fig.width=12, fig.height=6}
df_flux <- df |>
  mutate(
    flux_pos = pmax(CO2flux_mmol_m2_d, 0),   # above zero (outgassing)
    flux_neg = pmin(CO2flux_mmol_m2_d, 0)    # below zero (ingassing)
  )

# Per-season mid-point and label y-positions
ann_flux <- df_flux |>
  group_by(Season) |>
  summarise(
    DateTime = mean(DateTime,            na.rm = TRUE),
    y_outgas = max(CO2flux_mmol_m2_d,   na.rm = TRUE) * 0.6,
    y_ingas  = min(CO2flux_mmol_m2_d,   na.rm = TRUE) * 0.6,
    .groups  = "drop"
  )

p_flux <- ggplot(df_flux, aes(x = DateTime)) +
  night_rect(night_df) +
  geom_ribbon(
    aes(ymin = 0, ymax = flux_pos, fill = "Outgassing"),
    alpha = 0.45
  ) +
  geom_ribbon(
    aes(ymin = flux_neg, ymax = 0, fill = "Ingassing"),
    alpha = 0.45
  ) +
  geom_line(
    aes(y = CO2flux_mmol_m2_d),
    linewidth = 0.6, colour = "grey25"
  ) +
  geom_hline(yintercept = 0, linewidth = 0.8, colour = "black") +
  geom_text(
    data        = ann_flux,
    aes(x = DateTime, y = y_outgas, label = "Outgassing"),
    colour      = "#d73027", fontface = "bold",
    size        = 5,          # ~14 pt at figure size
    inherit.aes = FALSE
  ) +
  geom_text(
    data        = ann_flux,
    aes(x = DateTime, y = y_ingas, label = "Ingassing"),
    colour      = "#4575b4", fontface = "bold",
    size        = 5,
    inherit.aes = FALSE
  ) +
  scale_fill_manual(
    values = c(Outgassing = "#d73027", Ingassing = "#4575b4"),
    name   = NULL
  ) +
  scale_x_datetime(
    date_labels = "%b %d\n%H:%M",
    date_breaks = "4 hours",
    guide       = guide_axis(check.overlap = TRUE)
  ) +
  facet_wrap(~Season, scales = "free_x") +
  labs(
    x = NULL,
    y = expression(CO[2]~"flux (mmol/m"^2*"/d)")
  ) +
  theme_pub +
  theme(
    legend.position = "none",
    axis.text.x     = element_text(angle = 30, hjust = 1)
  )

print(p_flux)
save_plot(p_flux, "ts_CO2flux", width = 12, height = 6)
```

---

# Session Info

```{r session-info}
sessionInfo()
```